# ComplyAgent — Data Preparation

## Objective

The goal of this notebook is to prepare data for the machine learning
component of ComplyAgent.

The ML model will predict whether a customer is likely to experience
financial hardship.

This notebook covers the **Extract** stage of the data pipeline:

1. Load the raw Lending Club dataset
2. Explore the dataset
3. Identify hardship-related information
4. Select relevant pre-hardship features
5. Extract the selected features
6. Save the extracted dataset for the transformation stage

The raw dataset contains many columns that are not required for
hardship prediction, so only relevant features are extracted.

## 1. Import Libraries

We use Pandas and NumPy for data loading and basic data analysis.

In [1]:
import pandas as pd
import numpy as np

## 2. Load the Raw Lending Club Dataset

The original Lending Club dataset is stored in the `data/raw` folder.

The raw dataset is kept unchanged because it is our original source data.

In [2]:
raw_path = "../data/raw/accepted_2007_to_2018Q4.csv"

df = pd.read_csv(raw_path)

C:\Users\admin\AppData\Local\Temp\ipykernel_7824\2751486526.py:3: DtypeWarning: Columns (0: id, 19: desc, 59: verification_status_joint, 118: sec_app_earliest_cr_line, 129: hardship_type, 130: hardship_reason, 131: hardship_status, 134: hardship_start_date, 135: hardship_end_date, 136: payment_plan_start_date, 139: hardship_loan_status, 145: debt_settlement_flag_date, 146: settlement_status, 147: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_path)


## 3. Initial Dataset Inspection

First, we inspect the size, structure, columns, and basic statistics
of the raw dataset.

In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (1391383, 151)


In [4]:
df.head(2)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1391383 entries, 0 to 1391382
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(1), str(37)
memory usage: 1.6+ GB


In [6]:
df.describe()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,...,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,0.0,1.391367e+06,1.391367e+06,1.391367e+06,1.391367e+06,1.391367e+06,1.391367e+06,1.390433e+06,1.391366e+06,1.391366e+06,...,7241.0,7241.000000,7241.0,7241.000000,5765.000000,7241.000000,7241.000000,24966.000000,24966.000000,24966.000000
mean,NaN,1.523738e+04,1.523736e+04,1.523222e+04,1.293555e+01,4.483103e+02,7.803913e+04,1.913201e+01,3.251359e-01,6.972766e+02,...,3.0,147.131012,3.0,13.825853,433.902624,11112.423109,191.820193,4963.871237,47.621388,13.072899
std,NaN,9.112042e+03,9.112043e+03,9.109838e+03,4.776151e+00,2.637027e+02,1.181315e+05,1.377123e+01,8.979835e-01,3.249400e+01,...,0.0,124.333220,0.0,9.648014,365.738126,7370.435951,192.724082,3643.772249,7.198830,7.993146
min,NaN,1.000000e+03,1.000000e+03,7.250000e+02,5.310000e+00,1.401000e+01,0.000000e+00,-1.000000e+00,0.000000e+00,6.600000e+02,...,3.0,0.640000,3.0,0.000000,1.920000,55.730000,0.010000,44.210000,0.450000,0.000000
25%,NaN,8.000000e+03,8.000000e+03,8.000000e+03,9.430000e+00,2.568900e+02,4.650000e+04,1.218000e+01,0.000000e+00,6.700000e+02,...,3.0,55.230000,3.0,6.000000,163.650000,5329.470000,45.290000,2184.295000,45.000000,6.000000
50%,NaN,1.327500e+04,1.327500e+04,1.325000e+04,1.259000e+01,3.825000e+02,6.500000e+04,1.818000e+01,0.000000e+00,6.900000e+02,...,3.0,113.030000,3.0,15.000000,335.250000,9546.560000,133.020000,4151.705000,45.000000,12.000000
75%,NaN,2.000000e+04,2.000000e+04,2.000000e+04,1.561000e+01,5.954100e+02,9.330000e+04,2.493000e+01,0.000000e+00,7.150000e+02,...,3.0,203.170000,3.0,22.000000,594.870000,15409.540000,283.770000,6806.490000,50.000000,18.000000
max,NaN,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.100000e+08,9.990000e+02,5.800000e+01,8.450000e+02,...,3.0,943.940000,3.0,32.000000,2680.890000,40306.410000,1407.860000,30000.000000,521.350000,118.000000


In [7]:
df.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

# 4. Feature Extraction

The raw Lending Club dataset contains many columns, but the ML model
does not need all of them.

We select features that represent information that could reasonably
be available **before hardship occurs**.

The selected features cover:

- Loan information
- Income and employment
- Financial pressure
- Credit history

`hardship_flag` is kept separately as the target variable.

In [8]:
required_features = [
    # Loan information
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "purpose",

    # Income and employment
    "annual_inc",
    "emp_length",
    "home_ownership",
    "verification_status",

    # Financial pressure
    "dti",
    "dti_joint",
    "annual_inc_joint",
    "revol_bal",
    "revol_util",
    "tot_cur_bal",
    "tot_hi_cred_lim",
    "total_rev_hi_lim",

    # Credit history
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "pub_rec",
    "pub_rec_bankruptcies",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "delinq_amnt",
    "chargeoff_within_12_mths",

    # Target
    "hardship_flag"
]

## 5. Verify Selected Features

Before extracting the data, we check that every selected column exists
in the raw dataset.

In [9]:
missing_features = [
    col for col in required_features
    if col not in df.columns
]

if missing_features:
    print("Missing columns:", missing_features)
else:
    print("All selected features are available.")

All selected features are available.


## 6. Extract the Selected Features

We create a new DataFrame containing only the selected features.

The original raw dataset is not modified.

In [11]:
df_ml = df[required_features].copy()

print("Extracted dataset shape:", df_ml.shape)

Extracted dataset shape: (1391383, 30)


In [12]:
df_ml.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,annual_inc,emp_length,home_ownership,...,inq_last_6mths,open_acc,total_acc,pub_rec,pub_rec_bankruptcies,collections_12_mths_ex_med,acc_now_delinq,delinq_amnt,chargeoff_within_12_mths,hardship_flag
0,3600.0,36 months,13.99,123.03,C,C4,debt_consolidation,55000.0,10+ years,MORTGAGE,...,1.0,7.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,N
1,24700.0,36 months,11.99,820.28,C,C1,small_business,65000.0,10+ years,MORTGAGE,...,4.0,22.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,N
2,20000.0,60 months,10.78,432.66,B,B4,home_improvement,63000.0,10+ years,MORTGAGE,...,0.0,6.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,N
3,35000.0,60 months,14.85,829.90,C,C5,debt_consolidation,110000.0,10+ years,MORTGAGE,...,0.0,13.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,N
4,10400.0,60 months,22.45,289.91,F,F1,major_purchase,104433.0,3 years,MORTGAGE,...,3.0,12.0,35.0,0.0,0.0,0.0,0.0,0.0,0.0,N


## 7. Save the Extracted Dataset

The selected features are saved as a new CSV file.

The raw Lending Club dataset remains unchanged.

This file will be used as the input for the **Transform** stage.

In [13]:
processed_path = "../data/processed/hardship_data.csv"

df_ml.to_csv(
    processed_path,
    index=False
)

print(f"Saved extracted dataset to: {processed_path}")

Saved extracted dataset to: ../data/processed/hardship_data.csv


In [14]:
print("Final shape:", df_ml.shape)

Final shape: (1391383, 30)
